# PINN elasticity: Colab runner

Runs the **committed** code from GitHub on a Colab GPU. Run the cells top to bottom.

1. GPU check
2. Fresh clone of the repository at `REF`
3. Install the project **without touching Colab's GPU-enabled PyTorch**, then check the environment
4. Test suite
5. GPU smoke test (tiny networks, a few dozen iterations; not an experiment)

Rules: never edit code or configs here (change them locally, commit, push, re-run this notebook). Never paste tokens into a cell. Clear outputs before saving a copy of this notebook back to the repository.

Setup details: `docs/colab_setup.md` in the repository.

In [ ]:
# 1. GPU check. If this fails: Runtime -> Change runtime type -> GPU, then re-run.
!nvidia-smi --query-gpu=name,driver_version,memory.total --format=csv || echo "NO GPU: Runtime -> Change runtime type -> GPU"

In [ ]:
# 2. Fresh clone of the committed code. REF may be a branch, a tag or a full commit SHA.
REPO_URL = "https://github.com/aavig23/PINN-elasticity-2.git"
REF = "main"
REPO_DIR = "/content/PINN-elasticity-2"

!rm -rf {REPO_DIR} && git clone --quiet {REPO_URL} {REPO_DIR} && git -C {REPO_DIR} checkout --quiet {REF}
%cd {REPO_DIR}
!git log -1 --format="commit %H%n        %s"
!test -z "$(git status --porcelain)" && echo "working tree clean" || echo "WARNING: local changes present"

In [ ]:
# 3. Install the project without its dependencies, so Colab's preinstalled CUDA torch is kept.
#    check_environment.py then installs only packages that are completely missing (never torch)
#    and fails if there is no GPU or the project does not import.
!python -c "import torch; print('torch before install:', torch.__version__)"
!pip install --quiet --no-deps -e .
!python scripts/check_environment.py --install-missing --require-gpu
!python -c "import torch; print('torch after install: ', torch.__version__)"

In [ ]:
# 4. Test suite (CPU tests; about 10-30 s).
!python -m pytest -q

In [ ]:
# 5. GPU smoke test: both stages, tiny networks, a few dozen iterations. Not an experiment.
#    Writes to outputs/smoke/colab/ (git-ignored, lost when the runtime ends).
!python scripts/smoke_test.py --label colab

## Next

Running real experiments (`experiments/EXPnnn_*`), saving run folders and pushing results back to GitHub are added in later milestones (M5-M7).